# Аналіз E-Commerce платформи Olist

![Python](https://img.shields.io/badge/Python-3.14-blue?logo=python&logoColor=white)
![PostgreSQL](https://img.shields.io/badge/PostgreSQL-16-336791?logo=postgresql&logoColor=white)
![pandas](https://img.shields.io/badge/pandas-2.x-150458?logo=pandas&logoColor=white)
![Jupyter](https://img.shields.io/badge/Jupyter-Notebook-orange?logo=jupyter&logoColor=white)

Навчальний аналітичний проєкт на основі реальних даних бразильського маркетплейсу **Olist** — платформи, що з'єднує малих продавців з великими торговими майданчиками.

Мета — оцінити ефективність платформи: знайти найприбутковіші категорії, виявити проблеми з логістикою та зрозуміти, що впливає на задоволеність клієнтів.

## Датасет

**Джерело:** [Brazilian E-Commerce Public Dataset by Olist](https://www.kaggle.com/datasets/olistbr/brazilian-ecommerce) (Kaggle)  
**Період:** 2016–2018  
**Обсяг:** ~100k замовлень · ~100k відгуків · ~3k продавців · ~32k товарів  
**Формат:** 9 CSV-файлів     

| Файл | Що містить |
|---|---|
| `olist_orders_dataset.csv` | Замовлення, статуси, дати |
| `olist_order_items_dataset.csv` | Товари в замовленнях, ціна, фрахт |
| `olist_order_payments_dataset.csv` | Платежі, тип, сума, розстрочка |
| `olist_order_reviews_dataset.csv` | Відгуки клієнтів (оцінка 1–5) |
| `olist_products_dataset.csv` | Каталог товарів, категорія, розміри |
| `olist_sellers_dataset.csv` | Продавці, місто, штат |
| `olist_customers_dataset.csv` | Клієнти, місто, штат |
| `olist_geolocation_dataset.csv` | Геолокація за zip-кодом |
| `product_category_name_translation.csv` | Переклад категорій порт. → англ. |

## Аналітичні питання

1. Яка динаміка замовлень по місяцях — чи росте платформа?
2. Які топ-10 категорій генерують найбільший дохід?
3. Який середній час доставки по штатах і де найгірша логістика?
4. Як розподіляються оцінки відгуків і який % незадоволених клієнтів?
5. Хто топ-10 продавців за активністю і яка їхня середня оцінка?
6. Яка частка клієнтів повертається для повторної покупки?
7. Який середній чек по кожному типу оплати?
8. Чи впливає час доставки на оцінку клієнта?
9. Яка географія доходів платформи — які штати генерують найбільший GMV і яка середня цінність замовлення по регіонах?

## Технічний стек

| Інструмент | Для чого |
|---|---|
| `Python` + `pandas` | Очищення та трансформація даних |
| `SQLAlchemy` + `psycopg2` | Підключення та завантаження в PostgreSQL |
| `PostgreSQL` | Зберігання даних, SQL-аналіз |
| `matplotlib` + `seaborn` | Візуалізації |
| `scipy` | Статистичні тести |
| `Jupyter Notebook` | Документування аналізу |

## Детальний опис файлів датасету

---

### `olist_orders_dataset.csv`
Центральна таблиця датасету. Кожен рядок — одне замовлення.

| Колонка | Опис |
|---|---|
| `order_id` | Унікальний ідентифікатор замовлення (PK) |
| `customer_id` | ID клієнта, що зробив замовлення (FK → customers) |
| `order_status` | Статус замовлення: `delivered`, `shipped`, `canceled`, `processing` та ін. |
| `order_purchase_timestamp` | Дата і час оформлення замовлення |
| `order_approved_at` | Дата і час підтвердження оплати |
| `order_delivered_carrier_date` | Дата передачі замовлення перевізнику |
| `order_delivered_customer_date` | Фактична дата доставки клієнту |
| `order_estimated_delivery_date` | Очікувана дата доставки (обіцяна клієнту) |

---

### `olist_order_items_dataset.csv`
Товарний склад кожного замовлення. Одне замовлення може містити кілька позицій — тоді буде кілька рядків з однаковим `order_id`.

| Колонка | Опис |
|---|---|
| `order_id` | ID замовлення (FK → orders) |
| `order_item_id` | Порядковий номер товару всередині замовлення (1, 2, 3…) |
| `product_id` | ID товару (FK → products) |
| `seller_id` | ID продавця, який продає цей товар (FK → sellers) |
| `shipping_limit_date` | Крайній термін, до якого продавець має передати товар перевізнику |
| `price` | Ціна товару в бразильських реалах (BRL) |
| `freight_value` | Вартість доставки для цього товару |

---

### `olist_order_payments_dataset.csv`
Платіжні транзакції по замовленнях. Одне замовлення може мати кілька рядків — наприклад, якщо клієнт платив частково ваучером, а частково карткою.

| Колонка | Опис |
|---|---|
| `order_id` | ID замовлення (FK → orders) |
| `payment_sequential` | Порядковий номер платежу в рамках одного замовлення |
| `payment_type` | Тип оплати: `credit_card`, `boleto`, `voucher`, `debit_card` |
| `payment_installments` | Кількість розстрочок (для кредитної картки; 1 = без розстрочки) |
| `payment_value` | Сума транзакції в BRL |

---

### `olist_order_reviews_dataset.csv`
Відгуки клієнтів після отримання замовлення. Не всі клієнти залишають текст — поля коментарів часто порожні (~60%).

| Колонка | Опис |
|---|---|
| `review_id` | Унікальний ідентифікатор відгуку (PK) |
| `order_id` | ID замовлення, до якого належить відгук (FK → orders) |
| `review_score` | Оцінка від 1 до 5 (1 — найгірша, 5 — найкраща) |
| `review_comment_title` | Заголовок відгуку (може бути порожнім) |
| `review_comment_message` | Текст відгуку (може бути порожнім) |
| `review_creation_date` | Дата відправки запиту на відгук клієнту |
| `review_answer_timestamp` | Дата і час, коли клієнт заповнив відгук |

---

### `olist_products_dataset.csv`
Каталог товарів, що продаються на платформі.

| Колонка | Опис |
|---|---|
| `product_id` | Унікальний ідентифікатор товару (PK) |
| `product_category_name` | Назва категорії товару португальською мовою |
| `product_name_lenght` | Кількість символів у назві товару |
| `product_description_lenght` | Кількість символів в описі товару |
| `product_photos_qty` | Кількість фотографій товару в картці |
| `product_weight_g` | Вага товару в грамах |
| `product_length_cm` | Довжина товару в сантиметрах |
| `product_height_cm` | Висота товару в сантиметрах |
| `product_width_cm` | Ширина товару в сантиметрах |

---

### `olist_sellers_dataset.csv`
Інформація про продавців, зареєстрованих на платформі.

| Колонка | Опис |
|---|---|
| `seller_id` | Унікальний ідентифікатор продавця (PK) |
| `seller_zip_code_prefix` | Перші 5 цифр поштового індексу продавця |
| `seller_city` | Місто продавця |
| `seller_state` | Штат продавця (двобуквений код, напр. `SP`, `RJ`) |

---

### `olist_customers_dataset.csv`
Інформація про клієнтів. Важливо: кожне замовлення має унікальний `customer_id`, навіть якщо одна людина робила кілька замовлень — це особливість анонімізації датасету.

| Колонка | Опис |
|---|---|
| `customer_id` | Унікальний ID клієнта в рамках конкретного замовлення (PK) |
| `customer_unique_id` | Справжній унікальний ідентифікатор клієнта (один і той самий для всіх замовлень однієї людини) |
| `customer_zip_code_prefix` | Перші 5 цифр поштового індексу клієнта |
| `customer_city` | Місто клієнта |
| `customer_state` | Штат клієнта (двобуквений код) |

---

### `olist_geolocation_dataset.csv`
Координати для бразильських поштових індексів. Один zip-код може мати кілька записів — для аналізу беремо медіану координат.

| Колонка | Опис |
|---|---|
| `geolocation_zip_code_prefix` | Перші 5 цифр поштового індексу |
| `geolocation_lat` | Широта (latitude) |
| `geolocation_lng` | Довгота (longitude) |
| `geolocation_city` | Назва міста |
| `geolocation_state` | Штат (двобуквений код) |

---

### `product_category_name_translation.csv`
Словник для перекладу категорій товарів з португальської на англійську. Використовується при трансформації таблиці products.

| Колонка | Опис |
|---|---|
| `product_category_name` | Назва категорії португальською (ключ для join з products) |
| `product_category_name_english` | Назва категорії англійською |